In [1]:
import os
import sys
import pandas as pd
import argparse
from datetime import datetime
import pathlib


# Set up Python Path
```
為何會找不到路徑？

當開啟並執行這個 Jupyter Notebook 時，Python 預設只會從「Notebook 所在的目前資料夾」去尋找想要 import 的模組。

要載入的 Django 設定 (website_configs.settings) 以及 App 資料庫模型 (app_user_keyword_db.models)，它們都位在專案的上一層（也就是根目錄）。

如果不把上一層目錄（專案根目錄）手動加進搜尋路徑，Python 就不會知道要去外層找檔案。
```


In [2]:
# # Alternative 1: Use absolute path construction
# current_dir = os.path.abspath('')
# parent_dir = os.path.dirname(current_dir)
# sys.path.insert(0, parent_dir)

# # Alternative 2: Use pathlib for more modern path handling
# parent_path = pathlib.Path().absolute().parent
# sys.path.insert(0, str(parent_path))

# 新增：將上一層目錄加入 sys.path
parent_path = pathlib.Path().absolute().parent
sys.path.insert(0, str(parent_path))


In [3]:
# # Setup Django environment

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'website_configs.settings')
import django
django.setup()
# 重要：設定環境變數以允許在 Jupyter 的異步環境中執行同步操作
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"


## NewsData Model Structure

Let's first examine the structure of our NewsData model to understand its fields.

In [4]:
# Now we can import Django models
from app_user_keyword_db.models import NewsData

In [5]:
# Inspect the NewsData model fields
print('NewsData Model Fields:')
for field in NewsData._meta.get_fields():
    print(f'- {field.name}: {field.get_internal_type()}')

# Display total count of news entries
print(f'\nTotal news entries in database: {NewsData.objects.count()}')

NewsData Model Fields:
- item_id: CharField
- date: DateField
- category: CharField
- title: TextField
- content: TextField
- sentiment: FloatField
- top_key_freq: TextField
- tokens: TextField
- tokens_v2: TextField
- entities: TextField
- token_pos: TextField
- link: CharField
- photo_link: CharField

Total news entries in database: 0


# Import news data from csv

In [6]:
# Read CSV file
csv_file_path = '../app_user_keyword/dataset/udn_news_preprocessed.csv'
# csv_file_path = '../app_user_keyword/dataset/cna_news_preprocessed_12weeks.csv'
df = pd.read_csv(csv_file_path, sep='|')

# Process each row and create a NewsData object
for idx, row in df.iterrows():
    try:
        # Create or update NewsData object
        # This method tries to fetch an object with the given kwargs, 
        # and if it exists, updates the fields. If it doesn't exist, creates a new one.

        news_data, created = NewsData.objects.update_or_create(
            # Fields to look up the object查詢使用的欄位
            # 這些欄位的組合必須是唯一的，否則會導致錯誤
            # 這裡假設 item_id 是唯一的識別碼 通常是使用主鍵或唯一索引
            item_id=row['item_id'],
            # Fields to update or create with
            # 這些欄位是要更新或創建的欄位
            defaults={
                'date': datetime.strptime(row['date'], '%Y-%m-%d').date(),# Convert date string to datetime object
                'category': row['category'],
                'title': row['title'],
                'content': row['content'],
                'sentiment': row['sentiment'],
                #'summary': row['summary'],
                'top_key_freq': row['top_key_freq'],
                'tokens': row['tokens'],
                'tokens_v2': row['tokens_v2'],
                'entities': row['entities'],
                'token_pos': row['token_pos'],
                'link': row['link'],
                'photo_link': row['photo_link'] if row['photo_link'] != "" and not pd.isna(row['photo_link']) else None,
            }
        )
        if created:
            print(f"Created new NewsData object with item_id: {row['item_id']}")
        else:
            print(f"Updated existing NewsData object with item_id: {row['item_id']}")
    except Exception as e:
        print(f"Error at row {idx}: {e}")
        print(row)
# photo_link 欄位的值可能為以下幾種情況：
# 實際有值的 URL 字串
# 空字串 ("")
# Pandas NaN 值（當 CSV 檔案中該欄位為空時）
# None 值        

Created new NewsData object with item_id: udn_20260419_0_1
Created new NewsData object with item_id: udn_20260419_0_2
Created new NewsData object with item_id: udn_20260419_0_3
Created new NewsData object with item_id: udn_20260419_0_4
Created new NewsData object with item_id: udn_20260419_0_5
Created new NewsData object with item_id: udn_20260419_0_6
Created new NewsData object with item_id: udn_20260419_0_7
Created new NewsData object with item_id: udn_20260419_0_8
Created new NewsData object with item_id: udn_20260419_0_9
Created new NewsData object with item_id: udn_20260419_0_10
Created new NewsData object with item_id: udn_20260419_0_11
Created new NewsData object with item_id: udn_20260419_0_12
Created new NewsData object with item_id: udn_20260419_0_13
Created new NewsData object with item_id: udn_20260419_0_14
Created new NewsData object with item_id: udn_20260419_0_15
Created new NewsData object with item_id: udn_20260419_0_16
Created new NewsData object with item_id: udn_202

In [7]:
created

True

# Delete all data

In [9]:
# NewsData.objects.all().delete()

# Django ORM CRUD Operations

CRUD stands for Create, Read, Update, and Delete - the four basic operations for database management systems. In this tutorial, we'll explore how to perform these operations using Django ORM with the NewsData model.


# Query some data

In [8]:
# 1. 查詢所有資料，並限制只取前 5 筆 (Limit)
all_news = NewsData.objects.all()[:5]
for news in all_news:
    print(news.item_id, news.title)

udn_20260419_0_1 瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷
udn_20260419_0_2 伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
udn_20260419_0_3 中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！
udn_20260419_0_4 PO文「挑戰的一天」疑曝看診名字與年齡…嚇壞病人 不孕症醫師道歉了
udn_20260419_0_5 南市永康三民里「樹蛙巡守隊」每日巡查 意外救人一命


In [10]:
# 2. 條件過濾：查詢特定分類 (例如：政治) 的新聞
politics_news = NewsData.objects.filter(category="即時")[:5]
for news in politics_news:
    print(news.item_id, news.title)

udn_20260419_0_1 瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷
udn_20260419_0_2 伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
udn_20260419_0_3 中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！
udn_20260419_0_4 PO文「挑戰的一天」疑曝看診名字與年齡…嚇壞病人 不孕症醫師道歉了
udn_20260419_0_5 南市永康三民里「樹蛙巡守隊」每日巡查 意外救人一命


In [13]:
# 3. 多重條件過濾：查詢特定分類且標題包含特定關鍵字
tech_keyword_news = NewsData.objects.filter(category="社會", title__icontains="大甲")[:5]
for news in tech_keyword_news:
    print(news.item_id, news.title)

udn_20260419_4_8 影／大甲媽遶境彰化打起來偵查隊長鼻子被打斷 嫌犯持毒襲警擴大追查


In [14]:
# 3. 多重條件過濾：查詢特定多個分類 (例如：政治或科技) 且標題包含特定關鍵字
tech_keyword_news = NewsData.objects.filter(category__in=["社會", "地方"], title__icontains="大甲")[:5]
for news in tech_keyword_news:
    print(news.item_id, news.title)

udn_20260419_4_8 影／大甲媽遶境彰化打起來偵查隊長鼻子被打斷 嫌犯持毒襲警擴大追查
udn_20260419_5_3 大甲盃格鬥效應！媽祖今晚跨過濁水溪 部署2500人次警力護駕
udn_20260419_5_5 大甲媽遶境大打出手...偵查隊長鼻樑被打斷 彰檢聲押一嫌另追捕2人
udn_20260419_5_7 影／大甲媽經彰化有人打傷警 民俗專家：不知好歹的人會落災
udn_20260419_5_8 大甲媽遶境 彰化這處佛寺澎湃「清粥小菜」顏清標每年都來吃


In [15]:
# 4. 排除條件：查詢「非」特定分類的新聞
non_politics_news = NewsData.objects.exclude(category="要聞")[:5]
for news in non_politics_news:
    print(news.item_id, news.title)

udn_20260419_0_1 瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷
udn_20260419_0_2 伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
udn_20260419_0_3 中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！
udn_20260419_0_4 PO文「挑戰的一天」疑曝看診名字與年齡…嚇壞病人 不孕症醫師道歉了
udn_20260419_0_5 南市永康三民里「樹蛙巡守隊」每日巡查 意外救人一命


In [16]:
# 4. 排除條件：查詢「非」特定分類的新聞
non_politics_news = NewsData.objects.exclude(category__in=["社會", "即時"])[:5]
for news in non_politics_news:
    print(news.item_id, news.title)

udn_20260419_1_1 李四川要解決雙和塞車問題 新北市議員張嘉玲給建議
udn_20260419_1_2 大陸「惠台措施」釋利多？ 食藥署揭輸銷卡關：1568件申請僅15件核准
udn_20260419_1_3 商總明召集7大公會座談 觀光署籲業界抗拒大陸不當壓力
udn_20260419_1_4 「AI余登發」為軍購案發聲 賴總統指責不審總預算非真愛台灣
udn_20260419_1_5 趙少康拋軍購8100億 陳冠廷：善意的第一步、盼後續納商購及委製


In [17]:
from datetime import date, timedelta

In [19]:
# 5. 日期範圍過濾：查詢 2022 年 1 月份的新聞
jan_news = NewsData.objects.filter(date__gte=date(2026, 4, 19), date__lte=date(2026, 4, 19))

for news in jan_news:
    print(news.item_id, news.title, news.date)

udn_20260419_0_1 瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷 2026-04-19
udn_20260419_0_2 伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺 2026-04-19
udn_20260419_0_3 中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！ 2026-04-19
udn_20260419_0_4 PO文「挑戰的一天」疑曝看診名字與年齡…嚇壞病人 不孕症醫師道歉了 2026-04-19
udn_20260419_0_5 南市永康三民里「樹蛙巡守隊」每日巡查 意外救人一命 2026-04-19
udn_20260419_0_6 中職／生涯首度單場2長打 張祐銘透過大量練習挖掘力量 2026-04-19
udn_20260419_0_7 中職／雄鷹開季16戰10勝破隊史 陳文杰關鍵安打獲選MVP 2026-04-19
udn_20260419_0_8 李四川要解決雙和塞車問題 新北市議員張嘉玲給建議 2026-04-19
udn_20260419_0_9 國銀外幣存款登峰！兆元俱樂部大洗牌 臺銀飆上亞軍 2026-04-19
udn_20260419_0_10 兩個月超車一整年…外幣放款暴衝 AI點火銀行最猛開局 2026-04-19
udn_20260419_0_11 水利署稱水力發電未徵耗水費 翡翠水庫：多次支援台電「耗水入海」遭監院糾正 2026-04-19
udn_20260419_0_12 東大門夜市旁驚見醉漢持90公分劍恐嚇學生 警遭突襲掛彩壓制 2026-04-19
udn_20260419_0_13 UBA／澳洲家人來小巨蛋觀戰 飛尼克斯秀中文喊超開心 2026-04-19
udn_20260419_0_14 大陸破獲特大假酒跨省產銷鏈網絡 涉案金額人民幣2.6億元 2026-04-19
udn_20260419_0_15 台積電股價衝3000元？4年漲價計畫能解除「台海折價魔咒」嗎 2026-04-19
udn_20260419_0_16 商總擬動員七公協會表態 陸委會再喊話：勿成施壓工具 2026-04-19
udn_20260419_0_17 NBA／父親病後首度現身！唐斯下半場發威助尼克奪勝 賽後擁抱太催淚 2026-04-19
udn_20260419_0_18 蘋果MacBo

In [20]:
from django.db.models import Q, Max, F
# Complex filtering
# Get news from the last 7 days
latest_date = NewsData.objects.aggregate(max_date=Max('date'))['max_date']
start_date = latest_date - timedelta(days=3)
recent_news = NewsData.objects.filter(date__gte=start_date)
print(f"\nNews from the last 7 days: {recent_news.count()}")


News from the last 7 days: 140


In [22]:
from django.db.models import Count, Avg, Max, Min

# 6. 計算與統計 (Count, Aggregate)
total_count = NewsData.objects.count()
print(f"總資料筆數: {total_count}")

politics_count = NewsData.objects.filter(category="即時").count()
print(f"即時類新聞筆數: {politics_count}")

date_stats = NewsData.objects.aggregate(Max('date'), Min('date'))
print(f"最新新聞日期: {date_stats['date__max']}")
print(f"最舊新聞日期: {date_stats['date__min']}")

總資料筆數: 140
即時類新聞筆數: 20
最新新聞日期: 2026-04-19
最舊新聞日期: 2026-04-19


In [23]:
# 7. 分組計算 (Group By / Annotate)
# 統計每個分類 (category) 的新聞數量
category_counts = NewsData.objects.values('category').annotate(total=Count('item_id')).order_by('-total')

print("各分類新聞數量：")
for cat in category_counts:
    print(f"分類: {cat['category']}, 文章數: {cat['total']}")

各分類新聞數量：
分類: 運動, 文章數: 20
分類: 要聞, 文章數: 20
分類: 社會, 文章數: 20
分類: 產經, 文章數: 20
分類: 地方, 文章數: 20
分類: 即時, 文章數: 20
分類: 全球, 文章數: 20


In [24]:
# 8. 排序與取值 (Order By, Values)
# 找出特定欄位並排序
recent_news_titles = NewsData.objects.order_by('-date').values('title', 'date')[:5]

print("最新 5 篇新聞標題與日期：")
for news in recent_news_titles:
    print(f"日期: {news['date']}, 標題: {news['title']}")

print("\n")
# 取得最舊三篇的文字標題 List
title_list = NewsData.objects.order_by('date').values_list('title', flat=True)[:3]
print(f"純標題 List:\n{list(title_list)}")

最新 5 篇新聞標題與日期：
日期: 2026-04-19, 標題: 瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷
日期: 2026-04-19, 標題: 伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺
日期: 2026-04-19, 標題: 中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！
日期: 2026-04-19, 標題: PO文「挑戰的一天」疑曝看診名字與年齡…嚇壞病人 不孕症醫師道歉了
日期: 2026-04-19, 標題: 南市永康三民里「樹蛙巡守隊」每日巡查 意外救人一命


純標題 List:
['瓦拉米步道70歲婦人失聯9天 今尋獲遺體疑失足墜谷', '伊朗再關荷莫茲海峽 中印船舶似被迫折返、航運停擺', '中職／林鋅杰面對滿壘從沒自信到小小昇龍拳 葉總：表現真的很好！']


## Create (C) - Adding New Data

The 'Create' operation allows us to insert new records into the database. Here are different ways to create new NewsData objects.

In [25]:
# Method 1: Create using the constructor and save() method
new_news1 = NewsData(
    item_id="test_news_9001",  # 查詢條件 (通常為 PK 值)
    title="馬斯克蒞臨高科大",
    link="https://example.com/news1",
    category="測試類別",
    content="This is a sample content about technology news for demonstration.",
    date=datetime.now().date(),
    photo_link="https://example.com/image1.jpg",
    sentiment=0.75
)
new_news1.save()
# Uncomment to save: new_news1.save()

In [26]:
# Method 2: Create using objects.create() method
# 這種寫法更直接，它會同時實例化並寫入資料庫中，不需要額外呼叫 .save()
new_news2 = NewsData.objects.create(
    item_id="test_news_9002",  
    title="黃仁勳在台灣夜市品嘗美食",
    link="https://example.com/news2",
    category="測試類別",
    content="This is another sample content about technology news for demonstration.",
    date=datetime.now().date(),
    photo_link="https://example.com/image2.jpg",
    sentiment=0.88
)

print(f"成功使用 objects.create 建立資料: {new_news2.item_id} - {new_news2.title}")

成功使用 objects.create 建立資料: test_news_9002 - 黃仁勳在台灣夜市品嘗美食


## Update and create

In [27]:
from datetime import date

# 新增或更新 (update_or_create)
# 在爬蟲或匯入資料時非常實用。機制：透過指定的欄位（通常是 Primary Key 或 Unique 欄位，例如 item_id）尋找資料。
# 若找到資料，則更新 defaults 內的欄位；若找不到，則新建一筆資料。

news_obj, created = NewsData.objects.update_or_create(
    item_id="test_news_9999",  # 查詢條件 (通常為 PK 值)
    defaults={                 # 若新增或更新，要填入/覆蓋的內容
        "date": date.today(),
        "category": "系統測試",
        "title": "這是一篇測試 update_or_create 的新聞",
        "content": "這是為了展示 update_or_create 運作機制的測試內文。",
        "sentiment": 0.88
    }
)

if created:
    print(f"【成功建立新資料】: {news_obj.item_id} - {news_obj.title}")
else:
    print(f"【資料已存在，執行了更新】: {news_obj.item_id} - {news_obj.title}")

【成功建立新資料】: test_news_9999 - 這是一篇測試 update_or_create 的新聞


## Filter and delete

In [29]:
NewsData.objects.filter(category="測試類別").delete()

(0, {})

# Use Raw SQL to Create Article Table


## Plan: Use Raw SQL to Create Article Table

### Steps

1. **Import Django Connection Module** - Use Django's `connection` module to open a database cursor connection
2. **Create Article Table** - Execute raw SQL `CREATE TABLE` to build a table with fields: id (Primary Key), date, title, category, content
3. **Insert Test Data** - Execute `INSERT INTO` to populate the Article table with sample article records

### Implementation

The code in the next cell will:
- Open a cursor connection to the database
- Create the Article table with appropriate data types
- Insert three sample articles with dates, titles, categories, and content
- Verify the data was successfully written by querying all records

In [30]:
from django.db import connection

# 使用 Django 的資料庫連線執行原生 SQL
with connection.cursor() as cursor:
    
    # 先刪除表格
    cursor.execute("Drop Table article")
    
    # 建立 Article 資料表
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Article (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            date DATE,
            title TEXT,
            category TEXT,
            content TEXT
        )
    """)
    print("Article 資料表建立完成！")
    
    # 塞入幾筆測試資料
    cursor.execute("""
        INSERT INTO Article (date, title, category, content) 
        VALUES 
            ('2023-10-01', 'AI 發展趨勢', 'Tech', 'AI 正在改變世界...'),
            ('2023-10-02', '台北市長選舉預測', 'Politics', '今年的選情十分激烈...'),
            ('2023-10-03', '運動員奪下金牌', 'Sports', '精彩的表現讓全場歡呼...')
    """)
    print("測試資料建立完成！")

# 驗證資料是否成功寫入
with connection.cursor() as cursor:
    cursor.execute("SELECT * FROM Article;")
    rows = cursor.fetchall()
    
    print("\n查詢結果:")
    for row in rows:
        print(row)

OperationalError: no such table: article